In [ ]:
import os
import cv2
import numpy as np
import zipfile
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.utils import to_categorical
import seaborn as sns
IMG_SIZE = 64  # resize all images to 64x64



In [ ]:
IMG_SIZE = 64  # resize all images to 64x64

source_folders = {
    "eyes/train/Close" : r"C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\train\Close",
    "eyes/train/Open"  : r"C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\train\Open",
    "eyes/val/Close"   : r"C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\val\Close",
    "eyes/val/Open"    : r"C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\val\Open",
    "eyes/test/Close"  : r"C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\test\Close",
    "eyes/test/Open"   : r"C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\test\Open",
    "yawn/yawn"        : r"C:\Users\semwa\Project_Bnat\archive (1)\data\yawn\yawn",
    "yawn/no_yawn"     : r"C:\Users\semwa\Project_Bnat\archive (1)\data\yawn\no yawn",
}

# Labels: Close=1(drowsy), Open=0(alert), yawn=1(drowsy), no_yawn=0(alert)
label_map = {
    "eyes/train/Close" : 1,
    "eyes/train/Open"  : 0,
    "eyes/val/Close"   : 1,
    "eyes/val/Open"    : 0,
    "eyes/test/Close"  : 1,
    "eyes/test/Open"   : 0,
    "yawn/yawn"        : 1,
    "yawn/no_yawn"     : 0,
}

In [ ]:
X = []
y = []

for key, path in source_folders.items():
    label = label_map[key]
    print(f"Loading: {key} ...")
    
    for file in os.listdir(path):
        file_path = os.path.join(path, file)
        img = cv2.imread(file_path)
        
        if img is None:
            continue
        
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))   # resize to 64x64
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)   # grayscale
        
        X.append(img)
        y.append(label)

X = np.array(X)
y = np.array(y)

print(f"\nTotal Images Loaded : {X.shape[0]}")
print(f"Image Shape         : {X.shape}")
print(f"Labels Shape        : {y.shape}")

In [ ]:
# Normalize pixel values 0-255 → 0-1
X = X / 255.0

# Flatten 64x64 image → 4096 values (ANN needs 1D input)
X = X.reshape(X.shape[0], -1)

print(f"After Flatten Shape : {X.shape}")  # (num_images, 4096)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train : {X_train.shape[0]} images")
print(f"Val   : {X_val.shape[0]} images")
print(f"Test  : {X_test.shape[0]} images")

In [ ]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(IMG_SIZE * IMG_SIZE,)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64,  activation='relu'),
    Dense(1,   activation='sigmoid')   # binary output: drowsy(1) or alert(0)
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs          = 20,
    batch_size      = 32,
    validation_data = (X_val, y_val)
)

print("Model Training Done")

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy : {test_acc * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Alert (0)', 'Drowsy (1)']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Alert', 'Drowsy'],
            yticklabels=['Alert', 'Drowsy'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
model.save("drowsiness_ann_model.h5")
print("Model Saved as drowsiness_ann_model.h5")